In [1]:
import torch

import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional
from model import Transformer
from tokenizers import Tokenizer

In [2]:
tokenize = Tokenizer.from_file("./tokenizer/tokenizer.json")

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
@dataclass
class CFG:
    max_seq_len: int = 512
    vocab_size: int = tokenize.get_vocab_size()
    dim: int = 512
    n_layers: int = 8
    n_heads_q: int = 8
    n_heads_kv: Optional[int] = 4
    dropout: float = 0.0
    multiple_of: int = 256
    ffn_multiplier: Optional[int] = None
    norm_eps: float = 1e-6
    init_std: float = 0.02
    state_dict_path: str = "./model_states_tiny_stories/best_model.pth"

In [6]:
model = Transformer(CFG())

In [7]:
state_dict = torch.load(CFG.state_dict_path, map_location=torch.device('cpu'))
model.load_state_dict(state_dict['model_state_dict'])
model.to(device)
print(f"loaded {sum(p.numel() for p in model.parameters())/1e6:.2f}M params\n")

loaded 41.56M params



In [8]:
EOS = 1

@torch.no_grad()
def generate(prompt, n=220, temperature=0.8, top_k=50,seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    model.eval()
    tokens = [EOS] if prompt==None else tokenize.encode(prompt).ids
    out = torch.tensor([tokens], device=device)
    for _ in range(n-len(tokens)):
        ctx = out[:, -CFG.max_seq_len:]
        pos = torch.arange(ctx.size(1), device=device).repeat(1, 1)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            logits = model(ctx, pos, None)
            logits = logits[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float('Inf')
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        out = torch.cat((out, next_token), dim=1)
        if next_token.item() == EOS:
            break
    return tokenize.decode(out.squeeze().tolist())

In [ ]:
prompts = [None, "Once upon a time, there was a little dragon who", "Lily and Tom found a shiny key in the garden."]
for i, p in enumerate(prompts):
    print("="*78)
    print(f"PROMPT: {p!r}")
    print("-"*78)
    print(generate(p, seed=i))
    print()

PROMPT: None
------------------------------------------------------------------------------
One day, a big cat went to play in the park. The cat saw a ball and wanted to play with it. The ball was red and shiny. The cat was very happy, and it picked up the ball with its mouth.
As the cat played with the ball, it began to slip on the grass. The ball flew into the air and landed in a tree. The cat was sad because it could not get the ball back. A small bird saw the cat and wanted to help.
The bird flew up to the tree and got the ball for the cat. The cat was very happy and said, "Thank you, bird!" But then, something unexpected happened. The bird dropped the ball by mistake. The cat looked at the bird and said, "I forgive you, bird. Let's play together now!" And they played with the ball all day long.

PROMPT: 'Once upon a time, there was a little dragon who'
------------------------------------------------------------------------------
Once upon a time, there was a little dragon who wan

In [ ]:
@torch.no_grad()
def batched_generate(prompts, n=220, temperature=0.8, top_k=50, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    model.eval()
    batch_size = len(prompts)
    tokens_list = [[EOS] if prompt==None else tokenize.encode(prompt).ids for prompt in prompts]
    max_len = max(len(tokens) for tokens in tokens_list)
    out = torch.full((batch_size, n), EOS, dtype=torch.long, device=device)
    dones =torch.tensor([False] * batch_size, device=device).unsqueeze(1)
    for i, tokens in enumerate(tokens_list):
        out[i, :len(tokens)] = torch.tensor(tokens, device=device)

    for token_pos in range(1,n):
        ctx = out[:, max(0,token_pos-CFG.max_seq_len):token_pos]
        pos = torch.arange(ctx.size(1), device=device).repeat(batch_size, 1)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            logits = model(ctx, pos, None)
            logits = logits[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float('Inf')
        probs = F.softmax(logits, dim=-1)
        next_tokens = torch.multinomial(probs, num_samples=1)
        if token_pos<max_len:
            mask_1 = out[:,token_pos] != EOS
            next_tokens[mask_1] =  out[:,token_pos:token_pos+1][mask_1]

        mask = next_tokens == EOS
        dones = dones | mask
        next_tokens[dones] = EOS

        out[:,token_pos:token_pos+1] = next_tokens
        #print(next_tokens)
        if dones.all():
            break

    return [tokenize.decode(out[i].tolist()) for i in range(batch_size)]

In [11]:
out = batched_generate(prompts, seed=42)
for i, output in enumerate(out):
    print("="*78)
    print(f"PROMPT: {prompts[i]!r}")
    print("-"*78)
    print(output)
    print()

PROMPT: None
------------------------------------------------------------------------------
Once upon a time, there was a chubby little dog named Spot. Spot liked to play outside in the mud. He would jump and run, and his fur would get very dirty.
One day, Spot saw a big puddle of mud. He was so happy! He ran to the puddle and started to play in it. He splashed and rolled around, getting his fur all dirty. Spot was having so much fun!
After playing in the mud, Spot was very dirty. His fur was white, and his ears were brown. Spot's mom called him inside for a bath. Spot went inside and took a bath. He got clean and felt a little better. Then, Spot went to bed, dreaming of more fun days in the mud.

PROMPT: 'Once upon a time, there was a little dragon who'
------------------------------------------------------------------------------
Once upon a time, there was a little dragon who lived in a big forest. He had many friends, but he was not happy. Every day, he would look at the sky and dr